# **BERT for Sarcasm Detection**

## Objective

This notebook implements a fine-tuned BERT model for sarcasm detection across multiple datasets:

- News Headlines Dataset
- Reddit SARC Dataset
- SemEval Irony Dataset

## Goals

- Fine-tune BERT for in-domain sarcasm detection (RQ1)
- Evaluate whether contextual information improves detection (RQ2)
- Assess cross-domain generalisation (RQ3)

## Workflow

1. Clone repository and install dependencies
2. Load preprocessed datasets
3. RQ1 — In-domain BERT (no context)
4. RQ2 — BERT with vs without context (Reddit)
5. RQ3 — Cross-domain generalisation
6. Full results summary

## 1. Setup

In [4]:
import os
import sys
import pandas as pd
import numpy as np
import torch

# Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")

# Clone the repository
if not os.path.exists("Sarcasm_detection_in_social_media"):
    !git clone https://github.com/palishiita/Sarcasm_detection_in_social_media.git
else:
    print("Repo already cloned")

# Point to src folder in cloned repo
sys.path.append("/content/Sarcasm_detection_in_social_media/src")

from deep_models import (
    get_bert_model,
    tokenize_data,
    make_dataloader,
    train_bert,
    predict_bert
)
from evaluation import evaluate

# Data comes from Drive, models saved to cloned repo
PROC_DIR   = "/content/drive/MyDrive/Sarcasm_detection_in_social_media/data/processed"
MODELS_DIR = "/content/Sarcasm_detection_in_social_media/models"
os.makedirs(MODELS_DIR, exist_ok=True)

# Verify GPU
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
else:
    print("WARNING: No GPU detected. Go to Runtime > Change runtime type > T4 GPU")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Repo already cloned
GPU available: True
Device: Tesla T4


## 2. Load Preprocessed Datasets

Datasets were cleaned and split in `02_preprocessing.ipynb`.
Each is loaded directly from the processed folder.

Reddit is subsampled to **20,000 train / 5,000 test** samples with
`random_state=42` to match the LSTM baseline notebook exactly,
ensuring fair comparison across models.

In [5]:
# Headlines
hl_train = pd.read_csv(os.path.join(PROC_DIR, "headlines_train.csv"))
hl_test  = pd.read_csv(os.path.join(PROC_DIR, "headlines_test.csv"))

# Reddit — subsample
rd_train = pd.read_csv(os.path.join(PROC_DIR, "reddit_train.csv"))
rd_test  = pd.read_csv(os.path.join(PROC_DIR, "reddit_test.csv"))

rd_train = rd_train.sample(20000, random_state=42)
rd_test  = rd_test.sample(5000,  random_state=42)

# SemEval
se_train = pd.read_csv(os.path.join(PROC_DIR, "semeval_train.csv"))
se_test  = pd.read_csv(os.path.join(PROC_DIR, "semeval_test.csv"))

print("Dataset sizes:")
print(f"  Headlines — train: {len(hl_train):>6}  test: {len(hl_test)}")
print(f"  Reddit    — train: {len(rd_train):>6}  test: {len(rd_test)}")
print(f"  SemEval   — train: {len(se_train):>6}  test: {len(se_test)}")

Dataset sizes:
  Headlines — train:  22895  test: 5724
  Reddit    — train:  20000  test: 5000
  SemEval   — train:   3052  test: 764


## 3. Create Reddit Context Variants

Two Reddit variants are created for RQ2:

- **No context** — target comment only
- **With context** — parent comment prepended with `[SEP]` separator

This mirrors the preprocessing approach used in the LSTM notebook.

In [6]:
# Without context
rd_train_no_context = rd_train.copy()
rd_test_no_context  = rd_test.copy()

# With context
rd_train_with_context = rd_train.copy()
rd_test_with_context  = rd_test.copy()

rd_train_with_context["text_with_context"] = (
    rd_train_with_context["context"].fillna("").astype(str)
    + " [SEP] "
    + rd_train_with_context["text"].astype(str)
)

rd_test_with_context["text_with_context"] = (
    rd_test_with_context["context"].fillna("").astype(str)
    + " [SEP] "
    + rd_test_with_context["text"].astype(str)
)

print("No-context sample:")
print(rd_train_no_context["text"].iloc[0])
print("\nWith-context sample:")
print(rd_train_with_context["text_with_context"].iloc[0])

No-context sample:
it's hard not face-planting when wing-suiting through tight spaces in just cause 3.

With-context sample:
i was actually thinking the same thing when i did it, you can kind of see how i started to go down but noped out of it [SEP] it's hard not face-planting when wing-suiting through tight spaces in just cause 3.


## 4. RQ1 — In-Domain BERT (No Context)

A fresh BERT model is fine-tuned separately on each dataset and
evaluated on the held-out test set of the same dataset.

This answers: **How effectively can BERT detect sarcasm in short informal text?**

Fine-tuned models are saved to disk for reuse in RQ3.

In [7]:
print("=" * 60)
print("RQ1: IN-DOMAIN BERT — NO CONTEXT")
print("=" * 60)

rq1_results = []

datasets_rq1 = [
    ("Headlines", hl_train,           hl_test,            "text"),
    ("Reddit",    rd_train_no_context, rd_test_no_context, "text"),
    ("SemEval",   se_train,            se_test,            "text"),
]

for dataset_name, train_df, test_df, text_col in datasets_rq1:
    print(f"\n---------- {dataset_name} ----------")

    # Fresh BERT for each dataset
    model, tokenizer = get_bert_model()

    # Tokenize
    train_enc = tokenize_data(train_df[text_col], tokenizer)
    test_enc  = tokenize_data(test_df[text_col],  tokenizer)

    # Dataloaders
    train_loader = make_dataloader(train_enc, train_df["label"], shuffle=True)
    test_loader  = make_dataloader(test_enc,  test_df["label"],  shuffle=False)

    # Fine-tune
    model = train_bert(model, train_loader, epochs=3)

    # Predict and evaluate
    y_pred = predict_bert(model, test_loader)
    y_true = test_df["label"].values

    result = evaluate(y_true, y_pred,
                      dataset_name=dataset_name,
                      model_name="BERT (no context)")
    rq1_results.append(result)

    # Save for RQ3
    save_path = os.path.join(MODELS_DIR, f"bert_{dataset_name.lower()}")
    model.save_pretrained(save_path)
    tokenizer.save_pretrained(save_path)
    print(f"Model saved to: {save_path}")

rq1_df = pd.DataFrame(rq1_results)
print("\n===== RQ1 SUMMARY =====")
print(rq1_df[["dataset", "model", "accuracy", "f1_macro", "f1_sarcastic"]].to_string(index=False))

RQ1: IN-DOMAIN BERT — NO CONTEXT

---------- Headlines ----------


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1/3 — Loss: 0.3881
Epoch 2/3 — Loss: 0.1781
Epoch 3/3 — Loss: 0.1202

BERT (no context) on SemEval
              precision    recall  f1-score   support

           0       0.97      0.93      0.95       383
           1       0.94      0.97      0.95       381

    accuracy                           0.95       764
   macro avg       0.95      0.95      0.95       764
weighted avg       0.95      0.95      0.95       764



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to: /content/Sarcasm_detection_in_social_media/models/bert_semeval

===== RQ1 SUMMARY =====
  dataset             model  accuracy  f1_macro  f1_sarcastic
Headlines BERT (no context)  0.928372  0.927980      0.922671
   Reddit BERT (no context)  0.724000  0.723996      0.723003
  SemEval BERT (no context)  0.952880  0.952868      0.953608


## RQ1 — In-Domain Sarcasm Detection Performance

### Results

| Dataset    | Accuracy | Macro F1 | Sarcastic F1 |
|------------|----------|----------|--------------|
| Headlines  | 0.9284   | 0.9280   | 0.9227       |
| Reddit     | 0.7240   | 0.7240   | 0.7230       |
| SemEval    | 0.9529   | 0.9529   | 0.9536       |


BERT achieved strong in-domain performance across all three datasets, confirming its
effectiveness as a sarcasm detection model when trained and evaluated within the same
domain. However, performance varied considerably across datasets, reflecting differences
in linguistic complexity and domain characteristics.

**News Headlines (Accuracy: 0.928, F1: 0.923)**

The Headlines dataset produced the second-highest accuracy. News headlines are short,
self-contained, and linguistically structured, making sarcasm detection relatively
straightforward. Sarcastic headlines typically exploit a clear incongruity between formal
tone and absurd or critical content — a pattern that BERT's contextual representations
are well-suited to capture. The balanced precision (0.91 non-sarcastic, 0.95 sarcastic)
and recall scores confirm that the model learned a robust decision boundary for this
domain.

**SemEval (Accuracy: 0.953, F1: 0.954)**

SemEval achieved the highest in-domain performance despite being annotated for irony
rather than sarcasm. Two factors likely contribute to this result. First, the dataset is
the smallest of the three (~4,800 samples), which means the test set is less diverse and
easier to saturate with a powerful model like BERT. Second, Twitter irony tends to rely
on explicit linguistic markers — exaggerated phrasing, rhetorical questions, and strong
sentiment — that BERT's pre-trained representations can identify reliably. The
marginally higher sarcastic F1 (0.9536) compared to non-sarcastic (0.9521) suggests the
model slightly favoured the positive class, consistent with the expressive nature of
ironic tweets.

**Reddit (Accuracy: 0.724, F1: 0.723)**

Reddit produced the weakest in-domain result, consistent with expectations given the
dataset's conversational and noisy character. Reddit sarcasm is community-specific,
context-dependent, and frequently relies on shared knowledge between participants in a
thread. Without access to the parent comment (no-context condition), the model was
limited to surface-level textual cues. Despite this, 0.72 accuracy represents a
meaningful improvement over chance (0.50) and outperforms the BiLSTM baseline (0.67),
demonstrating that BERT's pre-trained contextual representations provide a stronger
inductive bias for informal text than recurrent architectures.

**Key Finding — RQ1**

BERT is an effective in-domain sarcasm detector, particularly for structured and
expressive text. Performance degrades predictably in conversational domains where meaning
is context-dependent rather than self-contained within the target sentence.

## 5. RQ2 — Effect of Context (Reddit)

Two BERT models are trained on Reddit:

- One using only the target comment (`text`)
- One using the parent comment prepended to the target (`text_with_context`)

The no-context result is reused directly from RQ1 — no retraining needed.

This answers: **Does incorporating contextual information improve sarcasm detection?**

In [8]:
print("=" * 60)
print("RQ2: EFFECT OF CONTEXT — REDDIT ONLY")
print("=" * 60)

rq2_results = []

# Reuse no-context result from RQ1
print("\nReusing Reddit (no context) result from RQ1...")
rq1_reddit = [r for r in rq1_results if r["dataset"] == "Reddit"][0].copy()
rq1_reddit["model"] = "BERT (no context)"
rq2_results.append(rq1_reddit)

# Train with context
print("\n---------- Reddit WITH Context ----------")

model_ctx, tokenizer_ctx = get_bert_model()

train_enc_ctx = tokenize_data(
    rd_train_with_context["text_with_context"], tokenizer_ctx
)
test_enc_ctx = tokenize_data(
    rd_test_with_context["text_with_context"], tokenizer_ctx
)

train_loader_ctx = make_dataloader(
    train_enc_ctx, rd_train_with_context["label"], shuffle=True
)
test_loader_ctx = make_dataloader(
    test_enc_ctx, rd_test_with_context["label"], shuffle=False
)

model_ctx = train_bert(model_ctx, train_loader_ctx, epochs=3)

y_pred_ctx = predict_bert(model_ctx, test_loader_ctx)
y_true_ctx = rd_test_with_context["label"].values

result_ctx = evaluate(y_true_ctx, y_pred_ctx,
                      dataset_name="Reddit",
                      model_name="BERT (with context)")
rq2_results.append(result_ctx)

rq2_df = pd.DataFrame(rq2_results)
print("\n===== RQ2 SUMMARY =====")
print(rq2_df[["model", "accuracy", "f1_macro", "f1_sarcastic"]].to_string(index=False))

RQ2: EFFECT OF CONTEXT — REDDIT ONLY

Reusing Reddit (no context) result from RQ1...

---------- Reddit WITH Context ----------


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1/3 — Loss: 0.6137
Epoch 2/3 — Loss: 0.4731
Epoch 3/3 — Loss: 0.3172

BERT (with context) on Reddit
              precision    recall  f1-score   support

           0       0.72      0.72      0.72      2510
           1       0.72      0.72      0.72      2490

    accuracy                           0.72      5000
   macro avg       0.72      0.72      0.72      5000
weighted avg       0.72      0.72      0.72      5000


===== RQ2 SUMMARY =====
              model  accuracy  f1_macro  f1_sarcastic
  BERT (no context)    0.7240  0.723996      0.723003
BERT (with context)    0.7206  0.720599      0.720208


## RQ2 — Effect of Contextual Information

### Results

| Model               | Accuracy | Macro F1 | Sarcastic F1 |
|---------------------|----------|----------|--------------|
| BERT (no context)   | 0.7240   | 0.7240   | 0.7230       |
| BERT (with context) | 0.7206   | 0.7206   | 0.7202       |

### Analysis

Incorporating contextual information — the parent Reddit comment prepended to the
target comment via a `[SEP]` token — produced no meaningful improvement and resulted
in a marginal performance decrease of 0.003 across all metrics. This finding is
consistent with results reported by the BiLSTM baseline and warrants careful
interpretation.

**Why context did not help**

The most likely explanation is that naive context concatenation introduces noise rather
than signal. The parent comment in Reddit SARC is authored by a different user and may
be stylistically and topically disconnected from the sarcastic reply. Simply appending
it to the input does not give the model an explicit mechanism to model the *relationship*
between context and reply — it merely extends the input sequence length, diluting the
features most relevant to sarcasm classification.

A secondary factor is data volume. With only 20,000 training samples, BERT may
lack sufficient examples to learn when and how contextual information is indicative of
sarcasm. More sophisticated architectures — such as CASCADE (Hazarika et al., 2018),
which explicitly models discourse relationships between utterances — have demonstrated
consistent context gains on Reddit, suggesting that the benefit of context is
architectural rather than purely informational.

It is also worth noting that the preprocessing pipeline strips URLs, mentions, and
non-ASCII characters from both the target and context text. This removes some of the
social cues (e.g. direct address, shared references) that make conversational context
informative in the first place.

**Key Finding — RQ2**

Naive context concatenation does not improve BERT's sarcasm detection on Reddit.
Meaningful exploitation of conversational context likely requires architectural
modifications that model the relationship between utterances explicitly, rather than
simple input-level concatenation.

## 6. RQ3 — Cross-Domain Generalisation

Each fine-tuned model from RQ1 is evaluated directly on the two
datasets it was never trained on — no additional training.

The drop in F1 score compared to in-domain performance (RQ1)
measures how much domain shift hurts generalisation.

This answers: **How well do sarcasm detection models generalise across domains?**

Cross-domain pairs evaluated:
- Headlines → Reddit, SemEval
- Reddit → Headlines, SemEval
- SemEval → Headlines, Reddit

In [9]:
from transformers import BertForSequenceClassification, BertTokenizer

print("=" * 60)
print("RQ3: CROSS-DOMAIN GENERALISATION")
print("=" * 60)

# Fixed test sets — identical to those used in RQ1
test_sets = {
    "Headlines": (hl_test,            "text"),
    "Reddit":    (rd_test_no_context, "text"),
    "SemEval":   (se_test,            "text"),
}

cross_domain_pairs = [
    ("Headlines", "Reddit"),
    ("Headlines", "SemEval"),
    ("Reddit",    "Headlines"),
    ("Reddit",    "SemEval"),
    ("SemEval",   "Headlines"),
    ("SemEval",   "Reddit"),
]

rq3_results = []

for train_domain, test_domain in cross_domain_pairs:
    print(f"\n---------- Train: {train_domain} → Test: {test_domain} ----------")

    # Reload fine-tuned model from RQ1 — no retraining
    model_path = os.path.join(MODELS_DIR, f"bert_{train_domain.lower()}")
    model     = BertForSequenceClassification.from_pretrained(model_path)
    tokenizer = BertTokenizer.from_pretrained(model_path)

    test_df, text_col = test_sets[test_domain]

    test_enc    = tokenize_data(test_df[text_col], tokenizer)
    test_loader = make_dataloader(test_enc, test_df["label"], shuffle=False)

    y_pred = predict_bert(model, test_loader)
    y_true = test_df["label"].values

    result = evaluate(y_true, y_pred,
                      dataset_name=f"{train_domain} → {test_domain}",
                      model_name="BERT (cross-domain)")
    rq3_results.append(result)

rq3_df = pd.DataFrame(rq3_results)
print("\n===== RQ3 SUMMARY =====")
print(rq3_df[["dataset", "model", "accuracy", "f1_macro", "f1_sarcastic"]].to_string(index=False))

RQ3: CROSS-DOMAIN GENERALISATION

---------- Train: Headlines → Test: Reddit ----------


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


BERT (cross-domain) on Headlines → Reddit
              precision    recall  f1-score   support

           0       0.50      0.91      0.65      2510
           1       0.50      0.09      0.16      2490

    accuracy                           0.50      5000
   macro avg       0.50      0.50      0.40      5000
weighted avg       0.50      0.50      0.40      5000


---------- Train: Headlines → Test: SemEval ----------


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


BERT (cross-domain) on Headlines → SemEval
              precision    recall  f1-score   support

           0       0.50      0.84      0.63       383
           1       0.51      0.17      0.26       381

    accuracy                           0.50       764
   macro avg       0.51      0.50      0.44       764
weighted avg       0.51      0.50      0.44       764


---------- Train: Reddit → Test: Headlines ----------


...


BERT (cross-domain) on Reddit → SemEval
              precision    recall  f1-score   support

           0       0.55      0.75      0.64       383
           1       0.61      0.38      0.47       381

    accuracy                           0.57       764
   macro avg       0.58      0.57      0.55       764
weighted avg       0.58      0.57      0.55       764


---------- Train: SemEval → Test: Headlines ----------


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


BERT (cross-domain) on SemEval → Headlines
              precision    recall  f1-score   support

           0       0.52      1.00      0.69      2997
           1       1.00      0.00      0.00      2727

    accuracy                           0.52      5724
   macro avg       0.76      0.50      0.34      5724
weighted avg       0.75      0.52      0.36      5724


---------- Train: SemEval → Test: Reddit ----------


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


BERT (cross-domain) on SemEval → Reddit
              precision    recall  f1-score   support

           0       0.50      1.00      0.67      2510
           1       0.25      0.00      0.00      2490

    accuracy                           0.50      5000
   macro avg       0.38      0.50      0.33      5000
weighted avg       0.38      0.50      0.34      5000


===== RQ3 SUMMARY =====
            dataset               model  accuracy  f1_macro  f1_sarcastic
 Headlines → Reddit BERT (cross-domain)  0.501800  0.401248      0.155879
Headlines → SemEval BERT (cross-domain)  0.503927  0.441735      0.255403
 Reddit → Headlines BERT (cross-domain)  0.511181  0.511076      0.518251
   Reddit → SemEval BERT (cross-domain)  0.568063  0.552607      0.469453
SemEval → Headlines BERT (cross-domain)  0.523934  0.344465      0.001466
   SemEval → Reddit BERT (cross-domain)  0.500800  0.334755      0.002398


## 7. Full Results Summary

All results consolidated across RQ1, RQ2, and RQ3.

In [10]:
print("=" * 60)
print("FULL RESULTS SUMMARY")
print("=" * 60)

print("\n--- RQ1: In-Domain Performance ---")
print(rq1_df[["dataset", "model", "accuracy", "f1_macro", "f1_sarcastic"]].to_string(index=False))

print("\n--- RQ2: Effect of Context (Reddit) ---")
print(rq2_df[["model", "accuracy", "f1_macro", "f1_sarcastic"]].to_string(index=False))

print("\n--- RQ3: Cross-Domain Generalisation ---")
print(rq3_df[["dataset", "model", "accuracy", "f1_macro", "f1_sarcastic"]].to_string(index=False))

print("\n--- COMBINED TABLE ---")
all_results = pd.concat([rq1_df, rq2_df, rq3_df], ignore_index=True)
print(all_results[["dataset", "model", "accuracy", "f1_macro", "f1_sarcastic"]].to_string(index=False))

FULL RESULTS SUMMARY

--- RQ1: In-Domain Performance ---
  dataset             model  accuracy  f1_macro  f1_sarcastic
Headlines BERT (no context)  0.928372  0.927980      0.922671
   Reddit BERT (no context)  0.724000  0.723996      0.723003
  SemEval BERT (no context)  0.952880  0.952868      0.953608

--- RQ2: Effect of Context (Reddit) ---
              model  accuracy  f1_macro  f1_sarcastic
  BERT (no context)    0.7240  0.723996      0.723003
BERT (with context)    0.7206  0.720599      0.720208

--- RQ3: Cross-Domain Generalisation ---
            dataset               model  accuracy  f1_macro  f1_sarcastic
 Headlines → Reddit BERT (cross-domain)  0.501800  0.401248      0.155879
Headlines → SemEval BERT (cross-domain)  0.503927  0.441735      0.255403
 Reddit → Headlines BERT (cross-domain)  0.511181  0.511076      0.518251
   Reddit → SemEval BERT (cross-domain)  0.568063  0.552607      0.469453
SemEval → Headlines BERT (cross-domain)  0.523934  0.344465      0.001466
   Sem

## RQ3 — Cross-Domain Generalisation

### Results

| Train → Test          | Accuracy | Macro F1 | Sarcastic F1 |
|-----------------------|----------|----------|--------------|
| Headlines → Reddit    | 0.5018   | 0.4012   | 0.1559       |
| Headlines → SemEval   | 0.5039   | 0.4417   | 0.2554       |
| Reddit → Headlines    | 0.5112   | 0.5111   | 0.5183       |
| Reddit → SemEval      | 0.5681   | 0.5526   | 0.4695       |
| SemEval → Headlines   | 0.5239   | 0.3445   | 0.0015       |
| SemEval → Reddit      | 0.5008   | 0.3348   | 0.0024       |

### Analysis

Cross-domain performance collapsed dramatically across all six transfer directions,
with accuracy ranging from 0.50 to 0.57 — barely above chance on a balanced binary
classification task. This represents a generalisation gap of up to 45 percentage points
compared to in-domain performance (e.g. SemEval in-domain: 0.953 vs SemEval → Reddit:
0.501).

**Pattern 1 — Universal failure of cross-domain transfer**

Every cross-domain pair produced near-chance accuracy, confirming that BERT fine-tuned
on one sarcasm domain does not generalise to another. This is a strong empirical result:
despite BERT's powerful pre-trained representations, fine-tuning causes the model to
overfit to domain-specific sarcasm patterns — vocabulary, syntax, register, and
annotation conventions — that do not transfer across text types. This finding is
consistent with the cross-domain generalisation gap identified as an open problem in
Chen et al. (2024) and Bodige et al. (2025).

**Pattern 2 — Reddit generalises best**

Reddit → Headlines (F1: 0.511) and Reddit → SemEval (F1: 0.553) are the strongest
cross-domain results. This is attributable to the size and diversity of the Reddit
training corpus. At 20,000 training samples drawn from a wide range of subreddits,
the Reddit-trained model is exposed to a greater variety of sarcastic expressions,
writing styles, and topic domains than models trained on the more homogeneous Headlines
or SemEval datasets. The Reddit → SemEval result (0.568) is particularly notable as the
highest cross-domain accuracy in the experiment, suggesting partial overlap between
conversational sarcasm patterns on Reddit and ironic expressions on Twitter.

**Pattern 3 — SemEval generalises catastrophically**

SemEval → Headlines and SemEval → Reddit both produced sarcastic F1 scores of
effectively zero (0.0015 and 0.0024 respectively). In both cases the model classified
nearly all test instances as non-sarcastic, despite achieving 0.953 accuracy in-domain.
This reflects extreme overfitting to Twitter-specific irony conventions. SemEval is the
smallest dataset (~3,800 training samples), which amplifies overfitting, and its irony
annotations capture a linguistic phenomenon that is only partially overlapping with
sarcasm as labelled in Reddit and Headlines. The preprocessing pipeline's removal of
hashtags, mentions, and emojis — markers that are particularly prominent in Twitter
irony — further reduces the transferable signal available to the model.

**Pattern 4 — Headlines generalises poorly to informal text**

Headlines → Reddit (sarcastic F1: 0.156) and Headlines → SemEval (sarcastic F1: 0.255)
demonstrate that a model trained on clean, formal journalistic text fails to detect
sarcasm in informal registers. News headline sarcasm relies on structural incongruity
within a tightly constrained format, while Reddit and Twitter sarcasm are expressed
through informal vocabulary, slang, and conversational tone that are entirely absent
from the Headlines training distribution.

**Key Finding — RQ3**

Cross-domain sarcasm detection remains a fundamentally unsolved problem even for
powerful transformer-based models. Fine-tuning BERT on a single domain produces a
domain-specialist rather than a domain-generalist. The direction of transfer matters
significantly: models trained on larger, more diverse corpora (Reddit) generalise
modestly better than those trained on smaller or more stylistically constrained datasets
(SemEval, Headlines). These findings reinforce the conclusion that sarcasm is not a
universal linguistic phenomenon with consistent surface-level markers, but rather a
highly domain-specific communicative strategy that requires domain-adapted modelling.

## Summary of Findings

| Research Question | Finding |
|---|---|
| RQ1 | BERT achieves strong in-domain performance on structured text (0.93–0.95) but degrades on conversational text (0.72). It outperforms the BiLSTM baseline across all datasets. |
| RQ2 | Naive context concatenation does not improve performance. The 0.003 drop with context suggests that simple input-level concatenation introduces noise rather than useful signal. |
| RQ3 | Cross-domain performance collapses to near-chance across all six transfer directions. Reddit-trained models generalise modestly better due to corpus diversity. SemEval-trained models fail catastrophically outside their domain. |

## Implications

The results collectively suggest three directions for future work. First, cross-domain
sarcasm detection would benefit from domain adaptation techniques such as adversarial
training or domain-invariant feature learning. Second, contextual modelling requires
architectural solutions beyond input concatenation — explicit discourse modelling or
hierarchical attention over utterance pairs is likely necessary. Third, the stark
contrast between SemEval's in-domain success and cross-domain failure highlights the
risk of over-reporting performance on small, stylistically homogeneous datasets, and
supports the call in the literature for more diverse, multi-domain evaluation benchmarks.